# Cardiac Patient Monitoring System

## Project Overview

This project analyzes a cardiovascular disease dataset using statistical analysis,
exploratory data analysis, supervised machine learning, feature engineering,
pipelines, and unsupervised learning.

The main objective is to explore patient-related features and build machine
learning models to classify whether cardiovascular disease is present.



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("cardio.csv", sep=";")

In [3]:
df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [4]:
df.shape

(70000, 13)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  int64  
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 6.9 MB


In [6]:
df.isnull().sum()

id             0
age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df["cardio"].value_counts()

cardio
0    35021
1    34979
Name: count, dtype: int64

In [9]:
df["cardio"].value_counts(normalize=True).mul(100).round(2)

cardio
0    50.03
1    49.97
Name: proportion, dtype: float64

### Cardio Analysis
The target variable is cardio, which represents the presence or absence
of cardiovascular disease.

- 0 → No cardiovascular disease
- 1 → Cardiovascular disease

In [10]:
df["age"].describe()

count    70000.000000
mean     19468.865814
std       2467.251667
min      10798.000000
25%      17664.000000
50%      19703.000000
75%      21327.000000
max      23713.000000
Name: age, dtype: float64

In [11]:
(df["age"] / 365.25).describe()

count    70000.000000
mean        53.302850
std          6.754967
min         29.563313
25%         48.361396
50%         53.943874
75%         58.390144
max         64.922656
Name: age, dtype: float64

### Age Analysis
The age feature is stored in days. The observed values correspond
approximately to ages between 30 and 65 years. No negative age values
were found.

In [12]:
df["gender"].describe()

count    70000.000000
mean         1.349571
std          0.476838
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          2.000000
Name: gender, dtype: float64

In [13]:
df["gender"].unique()

array([2, 1])

### Gender Analysis
The gender feature contains two encoded categories (1 and 2).
No unexpected gender categories were found.encoded as 1 = Female and 2 = Male

In [14]:
df["height"].describe()


count    70000.000000
mean       164.359229
std          8.210126
min         55.000000
25%        159.000000
50%        165.000000
75%        170.000000
max        250.000000
Name: height, dtype: float64

In [15]:
Q1_height = df["height"].quantile(0.25)
Q3_height = df["height"].quantile(0.75)

IQR_height = Q3_height - Q1_height

lower_height = Q1_height - 1.5 * IQR_height
upper_height = Q3_height + 1.5 * IQR_height

print("Q1:", Q1_height)
print("Q3:", Q3_height)
print("IQR:", IQR_height)
print("Lower Bound:", lower_height)
print("Upper Bound:", upper_height)

Q1: 159.0
Q3: 170.0
IQR: 11.0
Lower Bound: 142.5
Upper Bound: 186.5


In [16]:
height_outliers = (
    (df["height"] < lower_height) |
    (df["height"] > upper_height)
)

print("Number of height outliers:", height_outliers.sum())
print("Percentage:", round(height_outliers.mean() * 100, 2), "%")

Number of height outliers: 519
Percentage: 0.74 %


In [17]:
df.loc[height_outliers, "height"].value_counts().sort_index()

height
55      1
57      1
59      1
60      1
64      1
       ..
197     4
198    14
200     1
207     1
250     1
Name: count, Length: 65, dtype: int64

In [18]:
df.loc[df["height"] < 100, "height"].value_counts().sort_index()

height
55    1
57    1
59    1
60    1
64    1
65    2
66    1
67    3
68    2
70    3
71    1
72    1
74    1
75    2
76    1
80    1
81    1
91    1
96    1
97    1
98    1
99    1
Name: count, dtype: int64

### Height Analysis

The height feature ranges from 55 cm to 250 cm, with a mean of
approximately 164.36 cm.

The IQR method identified 519 observations (0.74%) as statistical
outliers. Additionally, 29 observations have a height below 100 cm.

In [19]:
df["weight"].describe()

count    70000.000000
mean        74.205690
std         14.395757
min         10.000000
25%         65.000000
50%         72.000000
75%         82.000000
max        200.000000
Name: weight, dtype: float64

In [20]:
print("Weight <= 0:", (df["weight"] <= 0).sum())

Weight <= 0: 0


In [21]:
Q1_weight = df["weight"].quantile(0.25)
Q3_weight = df["weight"].quantile(0.75)

IQR_weight = Q3_weight - Q1_weight

lower_weight = Q1_weight - 1.5 * IQR_weight
upper_weight = Q3_weight + 1.5 * IQR_weight

print("Q1:", Q1_weight)
print("Q3:", Q3_weight)
print("IQR:", IQR_weight)
print("Lower Bound:", lower_weight)
print("Upper Bound:", upper_weight)

Q1: 65.0
Q3: 82.0
IQR: 17.0
Lower Bound: 39.5
Upper Bound: 107.5


In [22]:
weight_outliers = (
    (df["weight"] < lower_weight) |
    (df["weight"] > upper_weight)
)

print("Number of weight outliers:", weight_outliers.sum())
print("Percentage:", round(weight_outliers.mean() * 100, 2), "%")

Number of weight outliers: 1819
Percentage: 2.6 %


In [23]:
df.loc[weight_outliers, "weight"].value_counts().sort_index()

weight
10.0     1
11.0     1
21.0     1
22.0     1
23.0     1
        ..
178.0    3
180.0    4
181.0    1
183.0    1
200.0    2
Name: count, Length: 93, dtype: int64

### Weight Analysis

The weight feature ranges from 10 kg to 200 kg, with a mean of
approximately 74.21 kg.

Using the IQR method, 1,819 observations (2.60%) were identified as
statistical outliers. The identified values range from very low weights
such as 10 kg to high values such as 200 kg.

In [24]:
df["cholesterol"].value_counts().sort_index()

cholesterol
1    52385
2     9549
3     8066
Name: count, dtype: int64

In [25]:
invalid_cholesterol = ~df["cholesterol"].isin([1, 2, 3])

print("Invalid cholesterol values:", invalid_cholesterol.sum())

Invalid cholesterol values: 0


### Cholesterol Analysis
The cholesterol feature represents the patient's cholesterol level.
It contains three ordinal categories:

- 1 : Normal
- 2 : Above normal
- 3 : Well above normal
  
The feature contains only the expected values (1, 2, and 3), with no invalid
categories detected.

In [26]:
df["gluc"].value_counts().sort_index()

gluc
1    59479
2     5190
3     5331
Name: count, dtype: int64

In [27]:
invalid_gluc = ~df["gluc"].isin([1, 2, 3])

print("Invalid glucose values:", invalid_gluc.sum())

Invalid glucose values: 0


### Glucose Analysis
The gluc feature represents the patient's glucose level.
It contains three ordinal categories:

- 1 : Normal
- 2 : Above normal
- 3 :  Well above normal

The feature contains only the expected values (1, 2, and 3), with no invalid
categories detected.

In [28]:
df["smoke"].value_counts().sort_index()

smoke
0    63831
1     6169
Name: count, dtype: int64

In [29]:
invalid_smoke = ~df["smoke"].isin([0, 1])

print("Invalid smoking values:", invalid_smoke.sum())

Invalid smoking values: 0


### Smoking Analysis

The smoke feature represents the patient's smoking status.
It is a binary variable with two categories:

- 0 : Non-smoker
- 1 : Smoker

The feature contains only the expected values (0 and 1), with no invalid
categories detected.

In [30]:
df["alco"].value_counts().sort_index()

alco
0    66236
1     3764
Name: count, dtype: int64

In [31]:
invalid_alco = ~df["alco"].isin([0, 1])

print("Invalid alcohol values:", invalid_alco.sum())

Invalid alcohol values: 0


### Alcohol Consumption Analysis

The alco feature represents the patient's alcohol consumption status.
It is a binary variable with two categories:

- 0 : Does not consume alcohol
- 1 : Consumes alcohol

The feature contains only the expected values (0 and 1), with no invalid
categories detected.

In [32]:
df["active"].value_counts().sort_index()

active
0    13739
1    56261
Name: count, dtype: int64

In [33]:
invalid_active = ~df["active"].isin([0, 1])

print("Invalid activity values:", invalid_active.sum())

Invalid activity values: 0


### Physical Activity Analysis

The active feature represents the patient's physical activity status.
It is a binary variable with two categories:

- 0: Not physically active
- 1: Physically active

The feature contains only the expected values (0 and 1), with no invalid
categories detected.

## Blood Pressure Analysis

The dataset contains two blood pressure features:

- ap_hi: Systolic blood pressure, representing the pressure during heart contraction.
- ap_lo: Diastolic blood pressure, representing the pressure during heart relaxation.

Both features are measured in mmHg.

In [34]:
df["ap_hi"].describe()

count    70000.000000
mean       128.817286
std        154.011419
min       -150.000000
25%        120.000000
50%        120.000000
75%        140.000000
max      16020.000000
Name: ap_hi, dtype: float64

In [35]:
print("ap_hi < 40:", (df["ap_hi"] < 40).sum())

df.loc[df["ap_hi"] < 40, "ap_hi"].value_counts().sort_index()

ap_hi < 40: 188


ap_hi
-150     1
-140     1
-120     2
-115     1
-100     2
 1       2
 7       1
 10      7
 11     28
 12     76
 13     15
 14     29
 15     12
 16      3
 17      3
 20      4
 24      1
Name: count, dtype: int64

In [36]:
print("ap_hi > 300:", (df["ap_hi"] > 300).sum())

df.loc[df["ap_hi"] > 300, "ap_hi"].value_counts().sort_index()

ap_hi > 300: 40


ap_hi
309      1
401      1
701      1
806      1
902      1
906      6
907      3
909      1
960      1
1110     1
1130     1
1202     1
1205     1
1300     2
1400     3
1409     1
1420     2
1500     1
1620     1
2000     1
11020    1
11500    1
13010    2
14020    4
16020    1
Name: count, dtype: int64

In [37]:
df["ap_lo"].describe()

count    70000.000000
mean        96.630414
std        188.472530
min        -70.000000
25%         80.000000
50%         80.000000
75%         90.000000
max      11000.000000
Name: ap_lo, dtype: float64

In [38]:
print("ap_lo < 40:", (df["ap_lo"] < 40).sum())

df.loc[df["ap_lo"] < 40, "ap_lo"].value_counts().sort_index()

ap_lo < 40: 59


ap_lo
-70     1
 0     21
 1      1
 6      2
 7      2
 8      2
 9      1
 10     7
 15     1
 20    15
 30     6
Name: count, dtype: int64

In [39]:
print("ap_lo > 300:", (df["ap_lo"] > 300).sum())

df.loc[df["ap_lo"] > 300, "ap_lo"].value_counts().sort_index()

ap_lo > 300: 953


ap_lo
585      1
602      1
700      1
708      2
709      2
        ..
9011     2
9100     1
9800     1
10000    3
11000    1
Name: count, Length: 62, dtype: int64

In [40]:
Q1_hi = df["ap_hi"].quantile(0.25)
Q3_hi = df["ap_hi"].quantile(0.75)

IQR_hi = Q3_hi - Q1_hi

lower_hi = Q1_hi - 1.5 * IQR_hi
upper_hi = Q3_hi + 1.5 * IQR_hi

print("Q1:", Q1_hi)
print("Q3:", Q3_hi)
print("IQR:", IQR_hi)
print("Lower Bound:", lower_hi)
print("Upper Bound:", upper_hi)

Q1: 120.0
Q3: 140.0
IQR: 20.0
Lower Bound: 90.0
Upper Bound: 170.0


In [41]:
Q1_lo = df["ap_lo"].quantile(0.25)
Q3_lo = df["ap_lo"].quantile(0.75)

IQR_lo = Q3_lo - Q1_lo

lower_lo = Q1_lo - 1.5 * IQR_lo
upper_lo = Q3_lo + 1.5 * IQR_lo

print("Q1:", Q1_lo)
print("Q3:", Q3_lo)
print("IQR:", IQR_lo)
print("Lower Bound:", lower_lo)
print("Upper Bound:", upper_lo)

Q1: 80.0
Q3: 90.0
IQR: 10.0
Lower Bound: 65.0
Upper Bound: 105.0


### Blood Pressure Analysis — Observation

The blood pressure features contain several extreme and implausible
recorded values.

For ap_hi (systolic blood pressure), 188 observations are below 40
and 40 observations are above 300.

For ap_lo (diastolic blood pressure), 59 observations are below 40
and 953 observations are above 300.

The extreme values include negative values, very low values, and
extremely high values such as 16020 for ap_hi and 11000 for ap_lo

## Data Cleaning

After completing the initial data exploration and data-quality analysis,
the data cleaning stage begins.

### Cleaning Blood Pressure Values

In [42]:
def clean_blood_pressure(df):
    valid_bp = (
        df["ap_hi"].between(40, 300) &
        df["ap_lo"].between(40, 300)
    )

    df.drop(index=df.index[~valid_bp], inplace=True)

    return df


clean_blood_pressure(df)
print("Dataset shape after cleaning:", df.shape)
print("Invalid ap_hi:", ((df["ap_hi"] < 40) | (df["ap_hi"] > 300)).sum())
print("Invalid ap_lo:", ((df["ap_lo"] < 40) | (df["ap_lo"] > 300)).sum())

Dataset shape after cleaning: (68775, 13)
Invalid ap_hi: 0
Invalid ap_lo: 0


### Cleaning Result

After removing records with invalid blood pressure values, the dataset
contains 68,775 observations and 13 columns.

No invalid values remain in either ap_hi or ap_lo.

In [43]:
def convert_age_to_years(df):
    df["age_years"] = df["age"] / 365.25
    return df


convert_age_to_years(df)


def remove_original_age(df):
    df.drop(columns=["age"], inplace=True)
    return df
remove_original_age(df)
    
df["age_years"].describe()



count    68775.000000
mean        53.290828
std          6.757383
min         29.563313
25%         48.342231
50%         53.938398
75%         58.381930
max         64.922656
Name: age_years, dtype: float64

### Removing Original Age Feature

After converting age from days to years and verifying the resulting
age_years feature, the original age column is no longer needed.

The original age feature is therefore removed to avoid keeping two
representations of the same information.